# Unsloth Finetuning - Local Version

This notebook is configured to run on your local machine. It imports credentials from the `api_credentials.py` file.

In [13]:
# Update packages but avoid NumPy 2.x
!pip install -U pip
# Remove NumPy upgrade - we'll install a specific version instead
!pip install -U transformers accelerate

ERROR: To modify pip, please run the following command:

C:\Users\Mohamed\anaconda3\python.exe -m pip install -U pip

  Using cached pip-25.2-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.2-py3-none-any.whl (1.8 MB)

Using cached pip-25.2-py3-none-any.whl (1.8 MB)
^C
^C


In [ ]:
# Fix NumPy compatibility issue - run this EARLY in the notebook
# This error happens when packages compiled with NumPy 1.x try to run with NumPy 2.x
print("Installing NumPy 1.24.3 for compatibility...")
!pip install numpy==1.24.3 -q
# Force reload
import importlib
import sys
if "numpy" in sys.modules:
    importlib.reload(sys.modules["numpy"])
    print("NumPy reloaded successfully!")

# Verify NumPy version
import numpy as np
print(f"NumPy version: {np.__version__}")

In [ ]:
# Install required packages
!pip install -qU transformers datasets optimum
!pip install -qU openai wandb
!pip install -qU json-repair
!pip install -qU faker
!pip install -qU vllm
!pip install -qU unsloth bitsandbytes accelerate peft trl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2025.9.9 requires datasets<4.0.0,>=3.4.1, but you have datasets 4.1.1 which is incompatible.
unsloth-zoo 2025.9.12 requires datasets<4.0.0,>=3.4.1, but you have datasets 4.1.1 which is incompatible.

unsloth 2025.9.9 requires datasets<4.0.0,>=3.4.1, but you have datasets 4.1.1 which is incompatible.
unsloth-zoo 2025.9.12 requires datasets<4.0.0,>=3.4.1, but you have datasets 4.1.1 which is incompatible.
  error: subprocess-exited-with-error
  
  × Building wheel for vllm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [2267 lines of output]
      C:\Users\Mohamed\AppData\Local\Temp\pip-build-env-1zzfaxss\overlay\Lib\site-packages\torch\_subclasses\functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pyt

## Import Credentials and Setup

In [1]:
# Import credentials from local file
import sys
import os

# Add the directory containing the credentials file to path
sys.path.append(os.path.dirname(os.path.abspath('__file__')))

try:
    from api_credentials import get_tokens, get_hf_token, get_wandb_token, get_openrouter_token
    
    # Set up tokens
    hf_token = get_hf_token()
    wandb_api_key = get_wandb_token()
    openrouter_api_key = get_openrouter_token()
    
    print("✅ Successfully loaded credentials")
except ImportError:
    print("❌ Error: Couldn't load api_credentials.py")
    print("Please ensure the file exists in the same directory as this notebook")
    # Provide fallback for testing
    hf_token = os.environ.get("HUGGINGFACE_TOKEN", "")
    wandb_api_key = os.environ.get("WANDB_API_KEY", "")
    openrouter_api_key = os.environ.get("OPENROUTER_API_KEY", "")

✅ Successfully loaded credentials


In [2]:
# Login to wandb and Hugging Face
import wandb

# Login to wandb
try:
    wandb.login(key=wandb_api_key)
    print("✅ Successfully logged in to Weights & Biases")
except Exception as e:
    print(f"❌ Error logging in to Weights & Biases: {e}")

# Login to Hugging Face
try:
    !huggingface-cli login --token {hf_token}
    print("✅ Successfully logged in to Hugging Face")
except Exception as e:
    print(f"❌ Error logging in to Hugging Face: {e}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Mohamed\_netrc
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Mohamed\_netrc
wandb: Currently logged in as: mohamedabozahrazxc (mohamedabozahrazxc-minufiya-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Currently logged in as: mohamedabozahrazxc (mohamedabozahrazxc-minufiya-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Successfully logged in to Weights & Biases

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.



Token is valid (permission: fineGrained).
The token `unsloth-token` has been saved to C:\Users\Mohamed\.cache\huggingface\stored_tokens
Your token has been saved to C:\Users\Mohamed\.cache\huggingface\token
Login successful.

The token `unsloth-token` has been saved to C:\Users\Mohamed\.cache\huggingface\stored_tokens
Your token has been saved to C:\Users\Mohamed\.cache\huggingface\token
Login successful.


✅ Successfully logged in to Hugging Face

The current active token is: `unsloth-token`



In [ ]:
# Import necessary libraries
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair
import torch

# Fix GPU detection for GTX 1650
print("=== GPU Detection and Setup ===")
print(f"PyTorch version: {torch.__version__}")

# Try to reinstall PyTorch with CUDA support if needed
if not torch.cuda.is_available():
    print("❌ CUDA not available. Your GTX 1650 is not being detected.")
    print("Installing PyTorch with CUDA support for your GTX 1650...")
    
    # Install PyTorch with CUDA 11.8 which is compatible with GTX 1650
    # Note: This will take some time to complete
    import sys
    if not sys.platform.startswith('win'):
        # Linux/Mac command
        !pip uninstall -y torch torchvision torchaudio
        !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    else:
        # Windows command
        !pip uninstall -y torch torchvision torchaudio
        !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    
    print("PyTorch reinstalled with CUDA support.")
    print("⚠️ Please restart the kernel/runtime for changes to take effect.")
    print("After restarting, run this cell again to check if GPU is detected.")
    
    # Force reload PyTorch to try detecting CUDA without restart
    import importlib
    importlib.reload(torch)

# Set up paths for local machine
# Change this to where your dataset is stored
data_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), "data")
os.makedirs(data_dir, exist_ok=True)

base_model_id = "Qwen/Qwen2-1.5B-Instruct"

# Improved GPU detection with error handling
try:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    if device == "cuda":
        print(f"✅ GPU detected successfully!")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
        gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU Memory: {gpu_mem_gb:.2f} GB")
        
        # Set appropriate dtype based on GPU capabilities
        if torch.cuda.get_device_capability()[0] >= 8:  # Ampere or newer
            torch_dtype = torch.bfloat16
            print("Using bfloat16 precision")
        else:
            torch_dtype = torch.float16
            print("Using float16 precision")
    else:
        torch_dtype = torch.float32
        print("Using float32 precision (CPU mode)")
        print("⚠️ If you have an NVIDIA GPU, please run the GPU diagnostic cell")
except Exception as e:
    print(f"Error during GPU detection: {e}")
    torch_dtype = torch.float32
    device = "cpu"
    print("Falling back to CPU mode due to error")

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

=== GPU Detection and Setup ===
PyTorch version: 2.8.0+cpu
❌ CUDA not available. Your GTX 1650 is not being detected.
Installing PyTorch with CUDA support for your GTX 1650...

PyTorch version: 2.8.0+cpu
❌ CUDA not available. Your GTX 1650 is not being detected.
Installing PyTorch with CUDA support for your GTX 1650...
Found existing installation: torch 2.8.0Found existing installation: torch 2.8.0

You can safely remove it manually.

Uninstalling torch-2.8.0:



  Successfully uninstalled torch-2.8.0
Found existing installation: torchvision 0.23.0
Uninstalling torchvision-0.23.0:
  Successfully uninstalled torchvision-0.23.0

Found existing installation: torchvision 0.23.0
Uninstalling torchvision-0.23.0:
  Successfully uninstalled torchvision-0.23.0
Found existing installation: torchaudio 2.4.1+cu118
Found existing installation: torchaudio 2.4.1+cu118
Uninstalling torchaudio-2.4.1+cu118:
  Successfully uninstalled torchaudio-2.4.1+cu118
Uninstalling torchaudio-2.4.1+cu118:
  Successfully uninstalled torchaudio-2.4.1+cu118


## Load and Prepare Dataset

In [4]:
# Define Pydantic models for data processing
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

StoryCategory = Literal[
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"
]

EntityType = Literal[
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"
]

class Entity(BaseModel):
    entity_value: str = Field(..., description="The actual name or value of the entity.")
    entity_type: EntityType = Field(..., description="The type of recognized entity.")

class NewsDetails(BaseModel):
    story_title: str = Field(..., min_length=5, max_length=300,
                             description="A fully informative and SEO optimized title of the story.")
    story_keywords: List[str] = Field(..., min_items=1,
                                      description="Relevant keywords associated with the story.")
    story_summary: List[str] = Field(
                                    ..., min_items=1, max_items=5,
                                    description="Summarized key points about the story (1-5 points)."
                                )
    story_category: StoryCategory = Field(..., description="Category of the news story.")
    story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                        description="List of identified entities in the story.")

class TranslatedStory(BaseModel):
    translated_title: str = Field(..., min_length=5, max_length=300,
                                  description="Suggested translated title of the news story.")
    translated_content: str = Field(..., min_length=5,
                                    description="Translated content of the news story.")

In [5]:
# Specify the path to your dataset
raw_data_path = join(data_dir, "news-sample.jsonl")

if not os.path.exists(raw_data_path):
    print(f"❌ Dataset not found at: {raw_data_path}")
    print("Please ensure your dataset is in the correct location")
else:
    raw_data = []
    for line in open(raw_data_path, encoding="utf-8"):
        if line.strip() == "":
            continue

        raw_data.append(
            json.loads(line.strip())
        )

    random.Random(101).shuffle(raw_data)
    print(f"✅ Raw data loaded: {len(raw_data)} samples")
    
    # Preview the first sample
    if len(raw_data) > 0:
        print("\nFirst sample preview:")
        print(raw_data[0]['content'][:300] + "..." if len(raw_data[0]['content']) > 300 else raw_data[0]['content'])

✅ Raw data loaded: 2400 samples

First sample preview:
يواصل المعهد العربي في باريس استقبال زواره في معرض ما تقدمه فلسطين للعالم لإطلاعهم على الإرث الثقافي والفني للفلسطينيين؛ من خلال أعمال فنية لآمالهم وصور لواقعهم الأليم تحت الاحتلال. 
 ويرى رئيس المعهد جاك لانغ -الذي أُعيد انتخابه قبل أيام للدورة الرابعة- ما يحدث في غزة حاليا جراء العدوان الإسرائيلي ...


## Knowledge Distillation (Generate SFT Data)

In [ ]:
# Option to generate SFT data using OpenRouter API
import os
import requests
import json
from tqdm.auto import tqdm
from os.path import join

# Path to save generated data
os.makedirs(join(data_dir, "generated"), exist_ok=True)
save_to = join(data_dir, "generated", "sft.jsonl")

def try_fix_json(text: str):
    """Attempt to clean and parse JSON from text response."""
    try:
        text = text.strip()
        if text.startswith("```"):
            parts = text.split("```")
            if len(parts) >= 2:
                text = parts[1]
        text = text.replace("\n", " ").strip()
        return json.loads(text)
    except Exception:
        return None

# Ask user if they want to generate data or use existing file
generate_data = input("Generate SFT data? (yes/no, default: no): ").lower().strip() == "yes"

if generate_data and len(raw_data) > 0:
    # How many samples to process
    max_samples = input("How many samples to process? (default: 5): ")
    max_samples = int(max_samples) if max_samples.isdigit() else 5
    
    # Use a limited subset for testing
    processing_data = raw_data[:max_samples]
    print(f"Processing {len(processing_data)} samples...")
    
    # Process each story
    ix = 0
    for story in tqdm(processing_data):
        # ===== Extract details =====
        sample_details_extraction_messages = [
            {
                "role": "system",
                "content": "\n".join([
                    "You are an NLP data parser.",
                    "You will be provided with an Arabic text associated with a Pydantic schema.",
                    "Generate the output in the same story language.",
                    "You have to extract JSON details from text according to the Pydantic details.",
                    "Extract details as mentioned in text.",
                    "Do not generate any introduction or conclusion."
                ])
            },
            {
                "role": "user",
                "content": "\n".join([
                    "## Story:",
                    story['content'].strip(),
                    "",
                    "## Pydantic Details:",
                    json.dumps(NewsDetails.model_json_schema(), ensure_ascii=False),
                    "",
                    "## Story Details:",
                    "```json"
                ])
            }
        ]

        # Send request to OpenRouter
        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {openrouter_api_key}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": "deepseek/deepseek-v3.1-terminus",
                    "messages": sample_details_extraction_messages,
                    "temperature": 0.2,
                    "max_tokens": 512
                },
                timeout=60
            )
            result = response.json()
        except Exception as e:
            result = {"error": str(e)}

        llm_response = None
        llm_resp_dict = None

        if "choices" in result:
            llm_response = result["choices"][0]["message"]["content"]
            llm_resp_dict = parse_json(llm_response) if parse_json else try_fix_json(llm_response)

        # Save everything (success or fail)
        record = {
            "id": ix,
            "story": story['content'].strip(),
            "task": "Extract the story details into a JSON.",
            "output_scheme": json.dumps(NewsDetails.model_json_schema(), ensure_ascii=False),
            "status": "success" if llm_resp_dict else "failed",
            "response": llm_resp_dict if llm_resp_dict else llm_response if llm_response else result
        }

        with open(save_to, "a", encoding="utf8") as dest:
            dest.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

        ix += 1
        print(f"Processed {ix} stories ✅")

    print(f"\n✅ Finished! Dataset at: {save_to}")
else:
    print(f"Skipping data generation. Looking for existing SFT data at: {save_to}")
    if not os.path.exists(save_to):
        print(f"⚠️ No SFT data found at {save_to}")
        print("You'll need to generate data first or specify a different path.")

## Prepare Dataset for Finetuning

In [ ]:
!pip install --upgrade --force-reinstall numpy pandas


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.3.3 which is incompatible.

  Using cached numpy-2.3.3-cp312-cp312-win_amd64.whl.metadata (60 kB)

numba 0.59.1 requires numpy<1.27,>=1.22, but you have numpy 2.3.3 which is incompatible.


opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which is incompatible.opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which is incompatible.

...

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)



streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.5 which is incompatible.

  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)

streamlit 1.32.0 requires tenacity<9,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.


  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)

Using cached numpy-2.3.3-cp312-cp312-win_amd64.whl (12.8 MB)Using cached numpy-2.3.3-cp312-cp312-win_amd64.whl (12.8 MB)

Using cached pandas-2.3.2-cp312-cp312-win_amd64.whl (11.0 MB)Using cached pandas-2.3.2-cp312-cp312-win_amd64.whl (11.0 MB)

Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)

Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)

Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)


  Attempting uninstall: pytz  Attempting uninstall: pytz

    Found existing installation: pytz 2025.2    Found existing installation: pytz 2025.2

  

In [6]:
# Load and prepare the dataset for finetuning
import json
import pandas as pd
from datasets import Dataset
import os

# Check if SFT data exists in the standard location or the colab location
sft_data_path = join(data_dir, "data", "sft.jsonl")
if not os.path.exists(sft_data_path):
    sft_data_path = join(data_dir, "sft.jsonl")
    if not os.path.exists(sft_data_path):
        alt_path = input("Enter path to SFT data (press Enter to abort): ").strip()
        if alt_path:
            sft_data_path = alt_path
        else:
            print("No SFT data provided. Cannot continue with finetuning.")
            sft_data = []

if os.path.exists(sft_data_path):
    # Load the data
    sft_data = []
    with open(sft_data_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                sft_data.append(json.loads(line))

    # Check if "status" exists in the first record
    has_status = len(sft_data) > 0 and "status" in sft_data[0]

    if has_status:
        # Filter successful responses
        successful_data = [item for item in sft_data if item.get("status") == "success"]
    else:
        # If no "status" field, just keep all data
        successful_data = sft_data

    print(f"Total examples: {len(sft_data)}, Successful examples: {len(successful_data)}")

    # Prepare the data for finetuning
    chat_data = []

    for item in successful_data:
        # Ensure response is always a string
        if isinstance(item["response"], dict):
            assistant_response = json.dumps(item["response"], ensure_ascii=False)
        else:
            assistant_response = str(item["response"])

        chat_example = {
            "messages": [
                {"role": "system", "content": "You are a helpful assistant that follows instructions precisely."},
                {"role": "user", "content": f"{item.get('story', '')}\n\nTask: {item.get('task', '')}"},
                {"role": "assistant", "content": assistant_response}
            ]
        }
        chat_data.append(chat_example)

    # Create a HuggingFace dataset
    dataset_df = pd.DataFrame(chat_data)
    hf_dataset = Dataset.from_pandas(dataset_df)

    # Split dataset
    train_test_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = train_test_split["train"]
    eval_dataset = train_test_split["test"]

    print(f"Training examples: {len(train_dataset)}, Evaluation examples: {len(eval_dataset)}")
    
    # Preview a training example
    if len(train_dataset) > 0:
        print("\nTraining example preview:")
        example = train_dataset[0]
        for msg in example["messages"]:
            print(f"Role: {msg['role']}")
            print(f"Content: {msg['content'][:100]}..." if len(msg['content']) > 100 else msg['content'])
            print()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Mohamed\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Mohamed\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Mohamed\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\Mohamed\anaconda3\Lib\s

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Mohamed\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Mohamed\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\Mohamed\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\Mohamed\anaconda3\Lib\s

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Total examples: 2766, Successful examples: 2766
Training examples: 2489, Evaluation examples: 277

Training example preview:
Role: system
You are a helpful assistant that follows instructions precisely.

Role: user
Content: قالت خبيرة التغذية الألمانية زيلكه ريستماير إن التفاح يسبب لدى بعض الأشخاص استجابات تحسسية، مثل الحك...

Role: assistant
Content: {"story_title": "فوائد التفاح وأعراض حساسية التفاح", "story_keywords": ["التفاح", "حساسية", "صحة", "...



## Configure Model for Finetuning

In [ ]:
# Setup for finetuning - with compatibility check
import torch
import os

from os.path import join
import sys

# Check for compatible GPU
has_compatible_gpu = False
gpu_type = "none"

if torch.cuda.is_available():
    gpu_type = "cuda"
    has_compatible_gpu = True
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ Compatible NVIDIA GPU detected: {gpu_name}")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    gpu_type = "xpu"
    has_compatible_gpu = True
    print("✅ Compatible Intel GPU detected")
else:
    print("❌ No compatible GPU detected. Unsloth requires an NVIDIA or Intel GPU.")
    print("Available options:")
    print("1. Continue with standard HuggingFace Transformers (slower)")
    print("2. Use CPU for inference only (no training)")
    choice = input("Select an option (1/2): ").strip()
    
    if choice == "2":
        print("Switching to inference-only mode...")
        # Skip to inference section
        sys.exit("Please run the inference cells directly")

# Create output directory
output_dir = join(data_dir, "finetuned")
os.makedirs(output_dir, exist_ok=True)

# Check if we have prepared dataset
if 'train_dataset' not in locals() or len(train_dataset) == 0:
    print("⚠️ No training dataset available. Please prepare the dataset first.")
    sys.exit("Please run the dataset preparation cells first")

# Configure training based on GPU
if has_compatible_gpu:
    # Use Unsloth for optimized training
    try:
        from unsloth import FastLanguageModel
        from transformers import TrainingArguments
        from trl import SFTTrainer
        
        # Detect GPU capabilities
        if gpu_type == "cuda":
            gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            print(f"GPU Memory: {gpu_mem:.2f} GB")
            
            # Determine optimal settings based on GPU
            if gpu_mem > 16:  # High-end GPU
                batch_size = 4
                gradient_accum = 4
                use_4bit = True
                precision = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
            elif gpu_mem > 8:  # Mid-range GPU
                batch_size = 2
                gradient_accum = 8
                use_4bit = True
                precision = torch.float16
            else:  # Low-end GPU
                batch_size = 1
                gradient_accum = 16
                use_4bit = True
                precision = torch.float16
        else:  # Intel XPU
            batch_size = 2
            gradient_accum = 8
            use_4bit = True
            precision = torch.float16
        
        print(f"Using batch size: {batch_size}, gradient accumulation: {gradient_accum}")
        print(f"Precision: {precision}, 4-bit quantization: {use_4bit}")
        
        # Load base model with Unsloth
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_id,
            max_seq_length=2048,
            dtype=precision,
            load_in_4bit=use_4bit,
            token=hf_token
        )
        print("✅ Model loaded successfully with Unsloth")
        
        # Configure the model for LoRA finetuning
        model = FastLanguageModel.get_peft_model(
            model,
            r=16,  # LoRA rank
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ],
            lora_alpha=16,
            lora_dropout=0.05,
            bias="none",
            use_gradient_checkpointing=True,
            random_state=42,
        )
        print("✅ LoRA configuration applied")
        
        # Setup training arguments
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=3,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accum,
            gradient_checkpointing=True,
            optim="adamw_torch",
            logging_steps=10,
            save_strategy="steps",
            save_steps=100,
            learning_rate=2e-4,
            weight_decay=0.01,
            fp16=(precision == torch.float16),
            bf16=(precision == torch.bfloat16),
            max_grad_norm=0.3,
            warmup_ratio=0.03,
            group_by_length=True,
            lr_scheduler_type="constant",
            report_to="wandb",
            seed=42,
        )
        
        # Prepare the trainer
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            dataset_text_field="messages",
            args=training_args,
            packing=True,
            max_seq_length=2048,
        )
        print("✅ Trainer configured successfully with Unsloth")
        
    except Exception as e:
        print(f"❌ Error setting up Unsloth: {e}")
        has_compatible_gpu = False  # Fall back to standard approach
        
# Standard HuggingFace approach (without Unsloth)
if not has_compatible_gpu:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    from transformers import TrainingArguments, Trainer
    from trl import SFTTrainer
    
    print("Using standard HuggingFace Transformers for training (CPU mode)")
    
    # Configure for CPU training (small and efficient as possible)
    batch_size = 1
    gradient_accum = 16
    
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)
        
        # Load model in 8-bit to save memory
        model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            device_map="auto",  # Will use CPU
            token=hf_token
        )
        print("✅ Model loaded in CPU mode")
        
        # Configure LoRA
        peft_config = LoraConfig(
            r=16,
            lora_alpha=16,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        
        # Apply LoRA config
        model = get_peft_model(model, peft_config)
        print("✅ LoRA configuration applied")
        
        # Setup training arguments (lightweight CPU version)
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=2,  # Fewer epochs for CPU
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accum,
            optim="adamw_torch",
            logging_steps=10,
            save_strategy="epoch",
            learning_rate=2e-4,
            weight_decay=0.01,
            warmup_ratio=0.03,
            report_to="wandb",
            seed=42,
        )
        
        # Prepare the trainer
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            dataset_text_field="messages",
            args=training_args,
            max_seq_length=1024,  # Lower for CPU training
        )
        print("✅ Trainer configured for CPU mode (training will be slow)")
        print("⚠️ WARNING: Training on CPU will be extremely slow and might take days")
        print("Consider using Google Colab with GPU acceleration instead")
        
    except Exception as e:
        print(f"❌ Error setting up standard training: {e}")

❌ No compatible GPU detected. Unsloth requires an NVIDIA or Intel GPU.
Available options:
1. Continue with standard HuggingFace Transformers (slower)
2. Use CPU for inference only (no training)


## Train the Model

In [ ]:
# Train the model
if 'trainer' in locals():
    start_training = input("Start training? (yes/no, default: yes): ").lower().strip() != "no"
    if start_training:
        try:
            print("Starting training...")
            trainer.train()
            print("✅ Training complete!")
        except Exception as e:
            print(f"❌ Error during training: {e}")
    else:
        print("Training skipped.")
else:
    print("Trainer not configured. Cannot start training.")

## Save the Trained Model

In [ ]:
# Save the model
if 'model' in locals() and 'tokenizer' in locals():
    final_model_path = join(data_dir, "finetuned", "qwen-arabic-assistant-final")
    os.makedirs(final_model_path, exist_ok=True)

    save_model = input("Save the model? (yes/no, default: yes): ").lower().strip() != "no"
    if save_model:
        try:
            # Save the trained model
            model.save_pretrained(final_model_path)
            tokenizer.save_pretrained(final_model_path)
            print(f"✅ Model saved to {final_model_path}")
        except Exception as e:
            print(f"❌ Error saving model: {e}")
    else:
        print("Model saving skipped.")
else:
    print("No model to save. Please train the model first.")

## Test the Finetuned Model

In [ ]:
# Test the finetuned model with inference
from unsloth import FastLanguageModel
import torch
import os

final_model_path = join(data_dir, "finetuned", "qwen-arabic-assistant-final")

if os.path.exists(final_model_path):
    # Test inference
    test_model = input("Test the model? (yes/no, default: yes): ").lower().strip() != "no"
    
    if test_model:
        try:
            # Determine precision based on GPU capabilities
            if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
                model_precision = torch.bfloat16
            else:
                model_precision = torch.float16
    
            # Load the finetuned model for inference
            inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
                model_name=final_model_path,
                max_seq_length=2048,
                dtype=model_precision,
                load_in_4bit=torch.cuda.is_available()
            )
            print("✅ Model loaded for inference")
    
            # Function to test the model
            def test_model(story, task):
                messages = [
                    {"role": "system", "content": "You are a helpful assistant that follows instructions precisely."},
                    {"role": "user", "content": f"{story}\n\nTask: {task}"}
                ]
    
                prompt = inference_tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
    
                inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)
    
                outputs = inference_model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True
                )
    
                response = inference_tokenizer.decode(outputs[0], skip_special_tokens=True)
                assistant_response = response.split("ASSISTANT: ")[-1].strip()
                return assistant_response
    
            # Test with a sample from the dataset if available
            if 'sft_data' in locals() and len(sft_data) > 0:
                sample = sft_data[0]
                story = sample.get("story", "No story available")
                task = sample.get("task", "No task available")
    
                print("Testing finetuned model:")
                print(f"Story: {story[:100]}...")
                print(f"Task: {task}")
                print("\nModel Response:")
                response = test_model(story, task)
                print(response)
            else:
                # Allow user to input test data
                print("No test data available from dataset. Please provide test data:")
                test_story = input("Enter a story to test: ")
                test_task = input("Enter a task (e.g., 'Extract the story details into a JSON.'): ")
                
                if test_story and test_task:
                    print("\nModel Response:")
                    response = test_model(test_story, test_task)
                    print(response)
                else:
                    print("No test input provided.")
        except Exception as e:
            print(f"❌ Error during model testing: {e}")
    else:
        print("Model testing skipped.")
else:
    print(f"❌ Model not found at {final_model_path}. Please train and save the model first.")

## Push Model to Hugging Face Hub (Optional)

In [ ]:
# Optional: Push model to Hugging Face Hub
if 'model' in locals() and 'tokenizer' in locals():
    push_to_hub = input("Push model to Hugging Face Hub? (yes/no, default: no): ").lower().strip() == "yes"
    
    if push_to_hub:
        # Get user details
        hf_username = input("Enter your Hugging Face username: ")
        model_name = input("Enter model name (default: qwen-arabic-assistant): ") or "qwen-arabic-assistant"
        
        if hf_username:
            try:
                full_model_name = f"{hf_username}/{model_name}"
                print(f"Pushing model to {full_model_name}...")
                
                model.push_to_hub(full_model_name, token=hf_token)
                tokenizer.push_to_hub(full_model_name, token=hf_token)
                
                print(f"✅ Model pushed to hub: {full_model_name}")
                print(f"View at: https://huggingface.co/{full_model_name}")
            except Exception as e:
                print(f"❌ Error pushing to hub: {e}")
        else:
            print("No username provided. Cannot push to hub.")
    else:
        print("Skipping push to hub.")
else:
    print("No model available to push to hub.")

## GPU Diagnostic
Let's check why your GPU isn't being detected.

In [ ]:
# Diagnose GPU issues
import torch
import subprocess
import sys
import os

print("=== PyTorch GPU Diagnostic ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if hasattr(torch.version, 'cuda') else 'Not found'}")

if not torch.cuda.is_available():
    print("\n=== Troubleshooting ===")
    print("Your NVIDIA GTX 1650 should be compatible, but CUDA is not being detected.")
    
    # Check if NVIDIA drivers are installed
    try:
        if os.name == 'nt':  # Windows
            nvidia_smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
            print("\n=== NVIDIA-SMI Output ===")
            print(nvidia_smi.stdout if nvidia_smi.returncode == 0 else "NVIDIA driver not found or not working properly")
        else:  # Linux/Mac
            nvidia_smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
            print("\n=== NVIDIA-SMI Output ===")
            print(nvidia_smi.stdout if nvidia_smi.returncode == 0 else "NVIDIA driver not found or not working properly")
    except:
        print("Could not run nvidia-smi - NVIDIA drivers might not be installed")
    
    print("\n=== SOLUTION ===")
    print("1. Install the correct NVIDIA drivers from: https://www.nvidia.com/Download/index.aspx")
    print("2. Install PyTorch with CUDA support using:")
    print("   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
    print("3. Restart your computer after driver installation")
    
    # Offer to install the correct PyTorch version
    install_pytorch = input("\nAttempt to install PyTorch with CUDA support now? (yes/no): ").strip().lower()
    if install_pytorch == "yes":
        print("Installing PyTorch with CUDA support...")
        # Choose an appropriate CUDA version for GTX 1650 (CUDA 11.8 is generally compatible)
        !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
        print("\nPlease restart the kernel/notebook for changes to take effect")
        print("After restart, re-run this cell to verify GPU detection")
else:
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device capability: {torch.cuda.get_device_capability(torch.cuda.current_device())}")
    print("\n✅ Your GPU is properly detected! You can proceed with GPU-accelerated training.")